In [4]:
# import os
# os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# os.environ["PMIX_MCA_gds"] = "hash"

from qiskit_metal import designs, Dict
from qiskit_metal.qlibrary.tlines.meandered_grounded import RouteMeanderGrounded
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround

# Derived length for a 5 GHz quarter-wave (open-short) fundamental at
# THIS design's cpw_width/cpw_gap - see docstring above. Independently
# cross-checked via design_helper.DesignHelper's elliptic-integral CPW
# model to 1.1858e8/(4*5e9)*1000 = 5.9291mm, matching to <0.03%.
TOTAL_LENGTH_MM = 5.92905
GROUND_WIDTH_UM = 10
# Keep the strap this far from either physical end of the meander path,
# since a strap placed exactly at an endpoint is not physically meaningful.
EDGE_MARGIN_MM = 0.5

# Chip footprint. Deliberately small since this design has no TL/
# launchpads to accommodate - just enough substrate/airbox margin around
# the resonator itself. Re-run the baseline after changing these, same
# caveat as the half-wave design: ground-plane proximity and airbox
# height both measurably shift f1 (the schizo-resonator chip-shrink test
# found a 25% Y-volume cut moved f1 by only 0.045%, but that was a
# DIFFERENT geometry - unverified here until tested).
CHIP_SIZE_X_MM = 3.0
CHIP_SIZE_Y_MM = 1.5
CHIP_SIZE_Z_UM = 500
CHIP_CENTER_X_MM = 0.0
CHIP_CENTER_Y_MM = -0.7


def build_design(ground_pos_mm=None, total_length_mm=TOTAL_LENGTH_MM,
                  chip_size_x_mm=None, chip_size_y_mm=None,
                  chip_center_x_mm=None, chip_center_y_mm=None):
    """
    Build the bare quarter-wave resonator design (no TL, no launchpads).

    ground_pos_mm : float or None
        Position (mm) along the resonator path, measured from the SHORT
        end (sg1's pin), for a single grounding strap. If None, builds
        the UNGROUNDED baseline (plain RouteMeander, no strap) - this is
        the reference point to validate/calibrate TOTAL_LENGTH_MM against
        the true measured f1, same role as in the half-wave design.
    total_length_mm : float
        Physical path length of the meander, in mm.
    chip_size_x_mm, chip_size_y_mm, chip_center_x_mm, chip_center_y_mm :
        float or None
        Chip footprint overrides. None uses the CHIP_* module defaults.
    """
    chip_size_x_mm = CHIP_SIZE_X_MM if chip_size_x_mm is None else chip_size_x_mm
    chip_size_y_mm = CHIP_SIZE_Y_MM if chip_size_y_mm is None else chip_size_y_mm
    chip_center_x_mm = CHIP_CENTER_X_MM if chip_center_x_mm is None else chip_center_x_mm
    chip_center_y_mm = CHIP_CENTER_Y_MM if chip_center_y_mm is None else chip_center_y_mm
    design = designs.DesignPlanar({}, overwrite_enabled=True)

    design.chips.main.size.size_x = f'{chip_size_x_mm}mm'
    design.chips.main.size.size_y = f'{chip_size_y_mm}mm'
    design.chips.main.size.size_z = f'{CHIP_SIZE_Z_UM}um'
    design.chips.main.size.center_x = f'{chip_center_x_mm}mm'
    design.chips.main.size.center_y = f'{chip_center_y_mm}mm'

    design.variables['cpw_width'] = '20 um'
    design.variables['cpw_gap'] = '12.25 um'

    # OPEN end (x=L, per the docstring's boundary-condition convention:
    # short at x=0, open at x=L) - this is what makes it lambda/4, NOT
    # OpenToGround at both ends (that would be the half-wave design).
    # OPEN end
    otg1 = OpenToGround(design, 'otg1', options=dict(
        chip='main', pos_x='-0.2mm', pos_y='-40um', orientation=180,
        width='20um', gap='12.25um', termination_gap='12.25um'  # <-- Added dimensions
    ))
    
    # SHORT end 
    sg1 = ShortToGround(design, 'sg1', options=dict(
        chip='main', pos_x='0mm', pos_y='-1.35mm', orientation=-90,
        width='20um', gap='12.25um'  # <-- Added dimensions
    ))

    common_kwargs = dict(
        trace_width='20um',
        trace_gap='12.25um',
        total_length=f'{total_length_mm}mm',
        hfss_wire_bonds=False,
        fillet='99.9 um',
        lead=dict(start_straight='300um'),
        pin_inputs=Dict(
            start_pin=Dict(component='sg1', pin='short'),
            end_pin=Dict(component='otg1', pin='open')),
    )

    if ground_pos_mm is None:
        # Baseline: no grounding strap at all - plain quarter-wave resonator
        res1 = RouteMeander(design, 'resonator1', Dict(**common_kwargs))
    else:
        if not (EDGE_MARGIN_MM <= ground_pos_mm <= total_length_mm - EDGE_MARGIN_MM):
            raise ValueError(
                f"ground_pos_mm={ground_pos_mm} out of valid range "
                f"[{EDGE_MARGIN_MM}, {total_length_mm - EDGE_MARGIN_MM}]"
            )
        res1 = RouteMeanderGrounded(design, 'resonator1', Dict(
            **common_kwargs,
            ground_straps=dict(
                positions=[f'{ground_pos_mm}mm'],
                width=f'{GROUND_WIDTH_UM}um'),
        ))

    return design


def sweep_positions_mm(n_points=9, total_length_mm=TOTAL_LENGTH_MM,
                        edge_margin_mm=EDGE_MARGIN_MM):
    """Evenly spaced grounding-strap positions spanning the full resonator
    length (minus edge margins). Does NOT include the None (baseline) case
    - add that separately in the caller."""
    import numpy as np
    return [round(float(x), 4) for x in
            np.linspace(edge_margin_mm, total_length_mm - edge_margin_mm, n_points)]



In [5]:

TOTAL_LENGTH_MM = 5.92905 

d_baseline = build_design(ground_pos_mm=None, total_length_mm= TOTAL_LENGTH_MM)

In [ ]:
from qiskit_metal import MetalGUI
gui = MetalGUI(d_baseline)